In [ ]:
import os
from google.colab import userdata

# 1. Get the key from Colab Secrets
api_key = userdata.get("APIKEY")

# 2. Set it as an environment variable (Crucial step)
os.environ["HUGGINGFACEHUB_API_TOKEN"] = api_key


In [ ]:
!pip install dotenv streamlit langchain  langchain-core langchain-community langchain-huggingface pydantic  chromadb  faiss-cpu  tiktoken  langchain-community   langchain_chroma  youtube-transcript-api langchain_classic


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 107.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180

In [ ]:
from dotenv import load_dotenv
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint,HuggingFaceEndpointEmbeddings
import os
from youtube_transcript_api import YouTubeTranscriptApi,TranscriptsDisabled

from langchain_core.prompts import PromptTemplate
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

load_dotenv()

llm =HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    huggingfacehub_api_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")
)

model = ChatHuggingFace(llm = llm)


In [ ]:
video_id = "giT0ytynSqg"

yt = YouTubeTranscriptApi()

""" try:
except:
    pass """

transcript_list =yt.fetch(video_id=video_id,languages=["en"])

transcript = " ".join(chunk.text for chunk in transcript_list)

# we have to split it into chucks

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=None,
    keep_separator=True,
    is_separator_regex=False
)

chunks = splitter.create_documents([transcript])

print(chunks[0])

# there are 111 chucks

# print(len(chunks))


page_content='They call you the Godfather of AI. So, what would you be saying to people about their career prospects in a world of super intelligence? Train to be a plumber. Really? Yeah. Okay, I'm going to become a plumber. Geoffrey Hinton is the Nobel Prize-winning pioneer whose groundbreaking work has shaped AI and the future of humanity. Why do they call you the Godfather of AI? Because there weren't many people who believed that we could model AI on the brain so that it learned to do complicated things like recognize objects in images or even do reasoning. And I pushed that approach for 50 years. And then Google acquired that technology. And I worked there for 10 years on something that's now used all the time in AI. And then you left? Yeah. Why? So that I could talk freely at a conference. What did you want to talk about freely? How dangerous AI could be. I realized that these things will one day get smarter than us. And we've never had to deal with that. And if you want to know 

In [ ]:
embedding_model = HuggingFaceEndpointEmbeddings(
    model="intfloat/multilingual-e5-large",
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN")
)


In [ ]:
chroma = Chroma(
    embedding_function=embedding_model,
)

vector_store = chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=None
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
vector_store.get(include = ['embeddings','documents', 'metadatas'])


{'ids': ['ea11bdf8-686c-4af4-bb4e-a770d1940273',
  '3210fbba-84c0-4926-a3f8-5f697b2fd418',
  'f9190a84-a56b-4084-880e-6db878f57b11',
  '495eb52c-d06a-4c5d-84ab-f212c6402d01',
  '4df0b04e-278c-4e35-b9ca-e6a293e1c5e7',
  'a0ffb98d-0f16-404b-9389-a67d09f1de64',
  'e3ce3a6b-6d23-46f4-b9e3-7af8a40fa679',
  'fcc979fa-e13d-416c-8827-c3fb68bb27e1',
  '8c505b96-0c39-4f2b-aede-88e5527d2c6b',
  '59fa5a51-522a-48d2-bfcf-f8dfe9930317',
  '67a879d0-a985-4a16-a307-82bdc8fd7994',
  '33d03b6a-a0da-4384-ac25-9f66d2321bfc',
  '6379a037-f735-45c5-bfdc-8f1c5e7dd17c',
  '4c9de39d-e1bd-4e36-8ec7-6d711b557e9f',
  '7a6cbfb0-f2b4-49e1-8c6a-d9f6227c8640',
  'b74676c9-1ebf-4232-b12b-504131b53a89',
  '6f5fa5bb-ca5a-4492-b47f-f029457ec404',
  'ba09a2cd-1808-4c42-a679-5ccf65788451',
  '518051b9-7528-4b2e-9a01-a9d95ef40e34',
  'cf26fc42-90c6-4898-b966-f68b680c2f99',
  'b1e20428-f564-4634-9b79-a7722ea2b6fa',
  '6d08adbd-4c3f-4298-a8d2-d9424c20ddb7',
  '92d20eab-b349-4405-babc-9434ea0f4ea4',
  '0406bfc2-5877-468c-a342-

In [ ]:
print(vector_store.similarity_search_with_score(
    query="What is the danger part of ai",
    k = 2
))

[(Document(id='59fa5a51-522a-48d2-bfcf-f8dfe9930317', metadata={}, page_content='to increase or decrease a connection strength so as to do better whatever task you\'re trying to do, then we could learn incredible things cuz that\'s what we\'re doing now with artificial neural nets. It\'s just we don\'t know for real brains how they get that signal about whether to increase or decrease. As we sit here today, what are the big concerns you have around safety of AI? If we were to to list the the top couple that are really front of mind and that we should be thinking about. Um Can I have more than a couple? Go ahead. I\'ll write them all down and we\'ll go through them. Okay, first of all, I want to make a distinction between two completely different kinds of risk. There\'s risks that come from people misusing AI. Yeah. And that\'s most of the risks and all of the short-term risks. And then there\'s risks that come from AI getting super smart and suddenly it doesn\'t need us. Is that a real

In [ ]:
retriever = vector_store.as_retriever(search_type = "similarity",earch_kwargs={"k": 4})

In [ ]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEndpointEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x7ab58b151eb0>, search_kwargs={})

In [ ]:
retriever.invoke("godfather of ai")

[Document(id='ea11bdf8-686c-4af4-bb4e-a770d1940273', metadata={}, page_content="They call you the Godfather of AI. So, what would you be saying to people about their career prospects in a world of super intelligence? Train to be a plumber. Really? Yeah. Okay, I'm going to become a plumber. Geoffrey Hinton is the Nobel Prize-winning pioneer whose groundbreaking work has shaped AI and the future of humanity. Why do they call you the Godfather of AI? Because there weren't many people who believed that we could model AI on the brain so that it learned to do complicated things like recognize objects in images or even do reasoning. And I pushed that approach for 50 years. And then Google acquired that technology. And I worked there for 10 years on something that's now used all the time in AI. And then you left? Yeah. Why? So that I could talk freely at a conference. What did you want to talk about freely? How dangerous AI could be. I realized that these things will one day get smarter than u

In [ ]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [ ]:
question = "z"
retrieved_docs = retriever.invoke(question)
retrieved_docs

[Document(id='00b07151-1718-44a4-9a42-27ceed8f4cd6', metadata={}, page_content='So, there\'s no physiological won\'t start sweating. Yeah. But it might have all the same behavior and in that case I\'d say, "Yeah, it\'s having emotion It\'s got an emotion." So, it\'s going to have the same sort of cognitive thought and then it\'s going to act upon that cognitive thought. way, but without the physiological responses. And does that matter that it doesn\'t go red in the face and it\'s just a different I mean, that\'s a response to the it somewhat different from us. Yeah. For some things, the physiological aspects are very important like love. They\'re a long way from having love the same way we do. But I don\'t see why they shouldn\'t have emotions. So, I think what\'s happened is people have a model of how the mind works and what feelings are and what emotions are and their model is just wrong. What um what brought you to Google? You You worked at Google for about a decade, right? Yeah. W

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'So, there\'s no physiological won\'t start sweating. Yeah. But it might have all the same behavior and in that case I\'d say, "Yeah, it\'s having emotion It\'s got an emotion." So, it\'s going to have the same sort of cognitive thought and then it\'s going to act upon that cognitive thought. way, but without the physiological responses. And does that matter that it doesn\'t go red in the face and it\'s just a different I mean, that\'s a response to the it somewhat different from us. Yeah. For some things, the physiological aspects are very important like love. They\'re a long way from having love the same way we do. But I don\'t see why they shouldn\'t have emotions. So, I think what\'s happened is people have a model of how the mind works and what feelings are and what emotions are and their model is just wrong. What um what brought you to Google? You You worked at Google for about a decade, right? Yeah. What brought you there? I have a son who has learning difficulties and in order 

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [ ]:
final_prompt

StringPromptValue(text='\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don\'t know.\n\n      So, there\'s no physiological won\'t start sweating. Yeah. But it might have all the same behavior and in that case I\'d say, "Yeah, it\'s having emotion It\'s got an emotion." So, it\'s going to have the same sort of cognitive thought and then it\'s going to act upon that cognitive thought. way, but without the physiological responses. And does that matter that it doesn\'t go red in the face and it\'s just a different I mean, that\'s a response to the it somewhat different from us. Yeah. For some things, the physiological aspects are very important like love. They\'re a long way from having love the same way we do. But I don\'t see why they shouldn\'t have emotions. So, I think what\'s happened is people have a model of how the mind works and what feelings are and what emotions are and their mode

In [ ]:
answer = model.invoke(final_prompt)
print(answer.content)

I don't know


# Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel,RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import  StrOutputParser

In [51]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text


In [52]:
parallel_chain = RunnableParallel({
    "context": retriever | RunnableLambda(format_docs),
    "question": RunnablePassthrough()
})

In [53]:
parallel_chain.invoke("godfather of ai")

{'context': 'They call you the Godfather of AI. So, what would you be saying to people about their career prospects in a world of super intelligence? Train to be a plumber. Really? Yeah. Okay, I\'m going to become a plumber. Geoffrey Hinton is the Nobel Prize-winning pioneer whose groundbreaking work has shaped AI and the future of humanity. Why do they call you the Godfather of AI? Because there weren\'t many people who believed that we could model AI on the brain so that it learned to do complicated things like recognize objects in images or even do reasoning. And I pushed that approach for 50 years. And then Google acquired that technology. And I worked there for 10 years on something that\'s now used all the time in AI. And then you left? Yeah. Why? So that I could talk freely at a conference. What did you want to talk about freely? How dangerous AI could be. I realized that these things will one day get smarter than us. And we\'ve never had to deal with that. And if you want to kn

In [54]:
parser = StrOutputParser()

In [57]:
main_chain = parallel_chain | prompt | model | parser

In [58]:
main_chain.invoke("What are the solution that is discussed in this video to get save from the disadvantages of ai")

'The solutions discussed in this video to get safe from the disadvantages of AI are:\n\n1. Slowing down AI development, but it is believed that this will not happen due to competition between countries and companies.\n2. Making AI safe, which some people believe is possible, like Ilya, who has billions of dollars of investment due to his faith in his ability to create safe AI.\n3. Recognizing AI as an existential threat and putting enormous resources into trying to figure out how to develop AI that won\'t take over from humans.\n4. Considering having "cold storage" or a backup system for data in case the internet goes down.\n5. Regulating AI use, but the current regulations are not designed to deal with the threats from AI.\n\nHowever, the speaker seems skeptical that these solutions will be effective and expresses concern about the potential dangers of AI.'

In [59]:
main_chain.invoke("summarize this video")

'The video discusses the possibility of mass joblessness when super-intelligent AI arrives. The speaker mentions that it will be a long time before AI surpasses humans in physical manipulation, and suggests that jobs in areas like plumbing could be safer until humanoid robots show up. The speaker also talks about the dangers of advanced technology, including the potential for drones and autonomous weapons to be used for malicious purposes, and the risk of a cyber attack combined with AI.'